In [ ]:
# ============================================
# CELL 1 : INSTALL DEPENDENCIES
# ============================================

!apt-get update -qq
!apt-get install -y ffmpeg

!pip install -q \
openai-whisper \
faster-whisper \
transformers \
sentencepiece \
accelerate \
torch \
torchvision \
torchaudio \
librosa \
python-docx \
reportlab \
pydub \
moviepy \
pandas \
Pillow \
ipywidgets

print("✅ Installation Complete")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 87 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 14.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 M

In [ ]:
# ============================================
# CELL 2 : IMPORT LIBRARIES
# ============================================

import os
import gc
import re
import json
import shutil
import tempfile
import subprocess
import base64

from io import BytesIO

import librosa
import pandas as pd

import torch
import whisper

from faster_whisper import WhisperModel

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

from pydub import AudioSegment
from moviepy.editor import VideoFileClip

from docx import Document

from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import LETTER

from google.colab import files

print("✅ Imports Successful")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if 

✅ Imports Successful


In [ ]:
# ============================================
# CELL 3 : CONFIGURATION
# ============================================

TEMP_DIR = "Uploads"
ARCHIVE_DIR = "Archive"

os.makedirs(TEMP_DIR, exist_ok=True)
os.makedirs(ARCHIVE_DIR, exist_ok=True)

THREAT_FILE = "threat_words.json"

HISTORY_FILE = "history.json"

DEFAULT_THREATS = {

"bomb",
"gun",
"attack",
"kill",
"terror",
"explosion",
"weapon",
"hostage",
"shoot",
"murder"

}

if not os.path.exists(THREAT_FILE):

    with open(THREAT_FILE,"w") as f:

        json.dump(list(DEFAULT_THREATS),f)

print("✅ Configuration Loaded")

✅ Configuration Loaded


In [ ]:
# ============================================
# CELL 4 : UPLOAD FILES
# ============================================

from google.colab import files

uploaded = files.upload()

media_files = []

for filename in uploaded.keys():

    destination = os.path.join(TEMP_DIR, filename)

    with open(destination, "wb") as f:
        f.write(uploaded[filename])

    media_files.append(destination)

print("\nUploaded Files:")

for file in media_files:
    print(file)

Saving New Recording 3.m4a to New Recording 3.m4a

Uploaded Files:
Uploads/New Recording 3.m4a


In [ ]:
# ============================================
# CELL 5 : LOAD MODEL
# ============================================

MODEL_TYPE = "whisper"      # options: whisper or faster
MODEL_SIZE = "large"

if MODEL_TYPE == "whisper":

    model = whisper.load_model(MODEL_SIZE)

else:

    model = WhisperModel(
        MODEL_SIZE,
        device="cuda" if torch.cuda.is_available() else "cpu",
        compute_type="float16" if torch.cuda.is_available() else "int8"
    )

print("✅ Model Loaded")

100%|█████████████████████████████████████| 2.88G/2.88G [00:41<00:00, 74.4MiB/s]


✅ Model Loaded


In [ ]:
# ============================================
# CELL 6 : EXTRACT AUDIO
# ============================================

from pydub import AudioSegment

def extract_audio(media_path):

    audio_path = media_path + ".wav"

    audio = AudioSegment.from_file(media_path)

    audio = audio.set_frame_rate(16000)

    audio = audio.set_channels(1)

    audio.export(audio_path, format="wav")

    return audio_path

print("✅ Audio Extraction Ready")

✅ Audio Extraction Ready


In [ ]:
# ============================================
# CELL 7 : TRANSCRIPTION
# ============================================

def transcribe(audio_path):

    if MODEL_TYPE == "whisper":

        result = model.transcribe(audio_path)

        segments = result["segments"]

        transcript = result["text"]

    else:

        segments, info = model.transcribe(audio_path)

        segments = list(segments)

        transcript = " ".join([s.text for s in segments])

    return transcript, segments


all_results = []

for media in media_files:

    print("=" * 60)
    print("Processing:", os.path.basename(media))

    audio = extract_audio(media)

    transcript, segments = transcribe(audio)

    print(transcript[:500])

    all_results.append(
        {
            "file": media,
            "audio": audio,
            "transcript": transcript,
            "segments": segments
        }
    )

print("\n✅ Transcription Complete")

Processing: New Recording 3.m4a


  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



 Hi, Devith. I am sorry to inform you that I broke your toy and I know you loved it very much but now it's broken but I think I can get it fixed. Why? I mean, how do you break it? What happened? I was playing with it and it fell down and it broke into pieces but I think by applying glue we can get it back. Okay, but how much time will it take because I am sad right now because I am having a bad day. So how much time will it take to fix the toy? I don't really know and I am sorry for your loss. I

✅ Transcription Complete


In [ ]:
# ============================================
# CELL 8 : THREAT DETECTION
# ============================================

with open(THREAT_FILE, "r") as f:
    THREAT_WORDS = set(json.load(f))

def detect_threats(text):

    found = []

    lower = text.lower()

    for word in THREAT_WORDS:

        if word.lower() in lower:

            found.append(word)

    return sorted(list(set(found)))


for result in all_results:

    threats = detect_threats(result["transcript"])

    result["threats"] = threats

    print("\n")
    print("="*60)
    print(os.path.basename(result["file"]))

    if threats:
        print("🚨 Threat Words Found:")
        print(threats)
    else:
        print("✅ No Threats Found")



New Recording 3.m4a
✅ No Threats Found


In [ ]:
# ============================================
# CELL 9 : SEARCH TRANSCRIPT
# ============================================

def search_transcript(text, keyword):

    keyword = keyword.lower()

    matches = []

    sentences = re.split(r"[.!?\n]", text)

    for sentence in sentences:

        if keyword in sentence.lower():

            matches.append(sentence.strip())

    return matches


keyword = input("Enter word to search : ")

for result in all_results:

    print("\n")
    print("="*60)
    print(os.path.basename(result["file"]))

    matches = search_transcript(
        result["transcript"],
        keyword
    )

    if len(matches)==0:

        print("No Matches Found")

    else:

        print(f"{len(matches)} Matches Found\n")

        for m in matches:

            print("•",m)

Enter word to search : Toy


New Recording 3.m4a
2 Matches Found

• I am sorry to inform you that I broke your toy and I know you loved it very much but now it's broken but I think I can get it fixed
• So how much time will it take to fix the toy


In [ ]:
# ============================================
# CELL 10 : LOAD SUMMARIZATION MODEL
# ============================================

MODEL_NAME="google/flan-t5-large"

tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)

summary_model=AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device="cuda" if torch.cuda.is_available() else "cpu"

summary_model.to(device)

print("✅ Summary Model Loaded")

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Summary Model Loaded


In [ ]:
# ============================================
# CELL 11 : GENERATE SUMMARY
# ============================================

def summarize(text,max_words=100):

    prompt="summarize: "+text

    inputs=tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    output=summary_model.generate(
        **inputs,
        max_length=max_words,
        num_beams=4
    )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )


for result in all_results:

    print("\n")
    print("="*60)

    summary=summarize(result["transcript"])

    result["summary"]=summary

    print(summary)



Devith broke Devith's toy. Devith will apply glue to fix the toy.


In [ ]:
# ============================================
# CELL 12 : EXPORT RESULTS
# ============================================

from io import BytesIO

def save_txt(text, filename):

    with open(filename,"w",encoding="utf-8") as f:

        f.write(text)


def save_docx(text, filename):

    doc=Document()

    doc.add_heading("Transcript",1)

    doc.add_paragraph(text)

    doc.save(filename)


def save_pdf(text, filename):

    c=canvas.Canvas(filename,pagesize=LETTER)

    y=760

    for line in text.split("\n"):

        c.drawString(40,y,line[:100])

        y-=15

        if y<40:

            c.showPage()

            y=760

    c.save()


for result in all_results:

    base=os.path.splitext(os.path.basename(result["file"]))[0]

    save_txt(result["transcript"],base+"_transcript.txt")

    save_docx(result["transcript"],base+"_transcript.docx")

    save_pdf(result["transcript"],base+"_transcript.pdf")

print("✅ Export Complete")

✅ Export Complete


In [ ]:
# ============================================
# CELL 13 : GENERATE VIDEO CLIPS
# ============================================

import os
import subprocess

CLIP_DIR = "Video_Clips"
os.makedirs(CLIP_DIR, exist_ok=True)

def create_video_clips(video_path, clip_length=10):

    clips = []

    probe = subprocess.run(
        [
            "ffprobe",
            "-v","error",
            "-show_entries","format=duration",
            "-of","default=noprint_wrappers=1:nokey=1",
            video_path
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    duration = int(float(probe.stdout))

    for start in range(0, duration, clip_length):

        end = min(start + clip_length, duration)

        output = os.path.join(
            CLIP_DIR,
            f"{os.path.splitext(os.path.basename(video_path))[0]}_{start}_{end}.mp4"
        )

        subprocess.run([
            "ffmpeg",
            "-y",
            "-ss", str(start),
            "-i", video_path,
            "-t", str(clip_length),
            "-c", "copy",
            output
        ])

        clips.append(output)

    return clips

print("✅ Video Clip Generator Ready")

✅ Video Clip Generator Ready


In [ ]:
 # ============================================
# CELL 14 : SAVE HISTORY
# ============================================

history = []

for result in all_results:

    history.append({

        "file": os.path.basename(result["file"]),

        "transcript": result["transcript"],

        "summary": result.get("summary",""),

        "threats": result.get("threats",[])

    })

with open("history.json","w") as f:

    json.dump(history,f,indent=4)

print("✅ History Saved")

✅ History Saved


In [ ]:
# ============================================
# CELL 15 : CSV REPORT
# ============================================

rows=[]

for result in all_results:

    rows.append({

        "Filename":os.path.basename(result["file"]),

        "Threats":", ".join(result.get("threats",[])),

        "Summary":result.get("summary","")

    })

df=pd.DataFrame(rows)

df.to_csv("Speech_Report.csv",index=False)

print(df)

print("✅ CSV Generated")

              Filename Threats  \
0  New Recording 3.m4a           

                                             Summary  
0  Devith broke Devith's toy. Devith will apply g...  
✅ CSV Generated


In [ ]:
# ============================================
# CELL 16 : DOWNLOAD OUTPUTS
# ============================================

from google.colab import files
import glob

for file in glob.glob("*"):

    if file.endswith(".txt"):

        files.download(file)

    elif file.endswith(".docx"):

        files.download(file)

    elif file.endswith(".pdf"):

        files.download(file)

    elif file.endswith(".csv"):

        files.download(file)

    elif file.endswith(".json"):

        files.download(file)

print("✅ Downloads Started")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloads Started


In [ ]:
# ============================================
# CELL 17 : FINAL SUMMARY
# ============================================

print("="*70)

print("SWAROOP Speech Analytics Completed Successfully")

print("="*70)

print()

print("Processed Files :",len(all_results))

print()

for result in all_results:

    print("------------------------------------------")

    print("File :",os.path.basename(result["file"]))

    print("Transcript Length :",len(result["transcript"].split()))

    print("Threat Words :",len(result.get("threats",[])))

    print()

print("="*70)

print("Outputs Generated")

print()

print("✔ Transcript (.txt)")

print("✔ Transcript (.docx)")

print("✔ Transcript (.pdf)")

print("✔ Summary")

print("✔ Threat Analysis")

print("✔ CSV Report")

print("✔ History JSON")

print("✔ Video Clips")

print("="*70)

SWAROOP Speech Analytics Completed Successfully

Processed Files : 1

------------------------------------------
File : New Recording 3.m4a
Transcript Length : 113
Threat Words : 0

Outputs Generated

✔ Transcript (.txt)
✔ Transcript (.docx)
✔ Transcript (.pdf)
✔ Summary
✔ Threat Analysis
✔ CSV Report
✔ History JSON
✔ Video Clips
